# NOTEBOOK 05 – VĨNH CHI | CHƯƠNG 5: KẾT QUẢ & THẢO LUẬN

**Phụ trách:** 5.1–5.4 · GitHub Issue **#6** · Bảng kết quả Test hạn 24/11, Ch5 hoàn chỉnh 25/11/2026.

**Vai trò độc lập:** đánh giá lại cả Random Forest (Ngọc Lan), Dummy (Trúc Chi) và MLP (Tấn Lên) trên **cùng frozen Test**, đối soát các artifact mà không fit lại.

**Nhận từ Ngọc Lan:** RF pipeline/metadata Validation. **Nhận từ Trúc Chi:** `test_predictions.csv`, `experiment_metadata.json` và thông tin ngưỡng/cấu hình chọn trên Validation.

**Bàn giao Tấn Lên:** `reports/tables/model_metrics.csv`, hình PR Curve, Top-K và bản phân tích lỗi.

[Lý thuyết Ch5](../docs/theory/05_vinh_chi_ch5.md) · [Giao thức thực nghiệm](../docs/EXPERIMENT_PROTOCOL.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 5.1 Kiểm tra frozen Test và lineage của cả 3 mô hình

Vĩnh Chi **không huấn luyện hoặc tối ưu ngưỡng**. Thay vào đó:
- Đối chiếu `split_id`, số đơn và thứ tự `order_id` giữa manifest với Test predictions Ch4.
- Load RF do Ngọc Lan fit, Dummy do Trúc Chi fit và MLP do Tấn Lên fit (chỉ artifact nội bộ đã xác minh).
- **Tự gọi predict_proba(Test)** cho cả ba và so sánh với `test_predictions.csv` để phát hiện kết quả bàn giao không khớp.
- Tính lại metrics độc lập; khác biệt phải báo Tấn Lên trước khi làm báo cáo.

In [ ]:
import json, joblib
from sklearn.metrics import (
    average_precision_score, confusion_matrix,
    precision_score, recall_score, f1_score
)
from src.experiment import get_shared_splits, SAFE_FEATURES
pred_file = PROCESSED / "test_predictions.csv"
meta_file = PROCESSED / "experiment_metadata.json"
if not pred_file.exists() or not meta_file.exists():
    raise FileNotFoundError("Thiếu frozen Test và metadata: yêu cầu Trúc Chi bàn giao.")
pred = pd.read_csv(pred_file)
meta = json.loads(meta_file.read_text(encoding="utf-8"))
splits = get_shared_splits(PROCESSED)
test_frame = splits["test"]
assert meta["split_id"] == splits["metadata"]["split_id"]
assert meta["features"] == list(SAFE_FEATURES)
assert meta["model_owners"] == {
    "rf": "Ngọc Lan", "dummy": "Trúc Chi", "mlp": "Tấn Lên"
}
expected = {"order_id","y_true","p_dummy","p_rf","p_mlp","y_pred_champion"}
assert expected.issubset(pred.columns), expected.difference(pred.columns)
assert pred["order_id"].is_unique and len(pred) == len(test_frame)
assert np.array_equal(pred["order_id"].to_numpy(),
                      test_frame["order_id"].to_numpy()), "Sai thứ tự/mẫu Test"
assert np.array_equal(pred["y_true"].to_numpy().astype(int),
                      test_frame["is_late"].to_numpy().astype(int))
for name in ("dummy", "rf", "mlp"):
    assert pred["p_"+name].between(0,1).all()
print("Split, sample order_id và model owner hợp lệ:", meta["split_id"])
print("Số dòng Test:", len(pred))

## 5.1.1 Kiểm định dự đoán bằng artifact độc lập (KHÔNG FIT)

Đây là bước kiểm định bổ sung: Vĩnh Chi dùng model do Ngọc Lan và Trúc Chi bàn giao để tính lại xác suất từ Test. Nếu kết quả không trùng \`test_predictions.csv\`, không được công bố kết quả cho tới khi giải quyết.

In [ ]:
model_files = {
    "rf": ROOT / "models" / "ngoc_lan_random_forest.joblib",
    "dummy": ROOT / "models" / "dummy_truc_chi.joblib",
    "mlp": ROOT / "models" / "tan_len_mlp.joblib",
}
missing = [p.name for p in model_files.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Thiếu artifact model cần kiểm định: " + ", ".join(missing))
X_test = test_frame[list(SAFE_FEATURES)]
for name, path in model_files.items():
    model = joblib.load(path)  # Chỉ load artifact tin cậy từ nội bộ nhóm
    rescored = model.predict_proba(X_test)[:,1]
    assert np.allclose(
        pred["p_"+name].to_numpy(), rescored, atol=1e-7
    ), f"Model {name} không khớp predictions bàn giao"
print("Vĩnh Chi đã tính lại xác suất của RF, Dummy và MLP trên frozen Test.")

## 5.1.1 Bảng chỉ số và ma trận nhầm lẫn

**Diễn giải:** Precision gắn với cảnh báo sai, Recall gắn với đơn trễ bị bỏ sót, F1 cân bằng hai khía cạnh, AP đo độ tốt xếp hạng qua ngưỡng. Luôn báo **số lượng thực (TP/FP/FN/TN)**, không chỉ phần trăm.

Ngưỡng của mỗi mô hình phải là ngưỡng đã chọn từ Validation; Dummy là mức so sánh.

In [ ]:
actual = pred["y_true"].to_numpy().astype(int)
rows = []
for name in ["dummy", "rf", "mlp"]:
    probabilities = pred["p_"+name].to_numpy()
    threshold = float(meta["thresholds_from_val"][name])
    labels = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(actual, labels, labels=[0,1]).ravel()
    rows.append({
        "model":name, "threshold_val":threshold, "AP":average_precision_score(actual, probabilities),
        "precision":precision_score(actual, labels, zero_division=0),
        "recall":recall_score(actual, labels, zero_division=0),
        "f1":f1_score(actual, labels, zero_division=0),
        "TP":int(tp), "FP":int(fp), "FN":int(fn), "TN":int(tn), "test_n":len(actual)
    })
metrics = pd.DataFrame(rows).sort_values("AP", ascending=False)
display(metrics)
OUT_TABLES = ROOT / "reports" / "tables"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
metrics.to_csv(OUT_TABLES / "model_metrics.csv", index=False)
print("Bảng metric đã xuất (chỉ có giá trị thực nếu Ch4 đã chạy).")

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
fig, ax = plt.subplots(figsize=(7,5))
for name in ["dummy","rf","mlp"]:
    PrecisionRecallDisplay.from_predictions(actual, pred["p_"+name], name=name, ax=ax)
ax.set_title("Đường Precision–Recall trên Test – Olist")
ax.grid(alpha=0.2)
fig.tight_layout()
OUT_FIGS = ROOT / "reports" / "figures"
OUT_FIGS.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_FIGS / "pr_curve_test.png", dpi=160)
plt.show()

## 5.2 Phân tích lỗi theo phân khúc

Cần chỉ ra khu vực, giá trị đơn, người bán/nhóm hàng có những cảnh báo sai hoặc bị bỏ sót. Chỉ phân tích slice có **đủ số mẫu**, tránh rút kết luận mạnh từ 2–3 đơn.

Có thể nối Test predictions với file `orders_model_ready.csv` bằng `order_id` (validate one_to_one), nhưng không coi các cột hậu nghiệm thành input của model.

In [ ]:
source = PROCESSED / "orders_model_ready.csv"
if not source.exists():
    raise FileNotFoundError("Thiếu orders_model_ready.csv để phân tích lỗi theo phân khúc.")
features = pd.read_csv(source)
from src.contracts import assert_unique_orders
assert_unique_orders(features)
champion = meta["champion"]
threshold = float(meta["chosen_threshold"])
pred["chosen_pred"] = (pred["p_"+champion] >= threshold).astype(int)
joined = pred.merge(
    features[["order_id","customer_state","total_price","seller_count"]].copy(),
    on="order_id", how="left", validate="one_to_one"
)
joined["false_alarm"] = (joined["chosen_pred"] == 1) & (joined["y_true"] == 0)
joined["missed_late"] = (joined["chosen_pred"] == 0) & (joined["y_true"] == 1)
by_region = joined.groupby("customer_state", dropna=False, as_index=False).agg(
    n=("order_id","size"), false_alarms=("false_alarm","sum"),
    missed_late=("missed_late","sum"), actual_late=("y_true","sum")
).sort_values("n", ascending=False)
display(by_region.loc[by_region["n"] >= 30].head(20))
print("Các thống kê phân khúc phản ánh nhóm đơn delivered có nhãn, không khẳng định nhân quả.")

## 5.3 Đánh giá ý nghĩa vận hành: Top-K

Kịch bản: đội vận hành đủ khả năng theo dõi K đơn, cần chọn đơn có xác suất cao nhất. Tính **Precision@K**, **Recall@K**, **FP@K** và so sánh với cảnh báo theo ngưỡng.

K phải được chọn từ nhu cầu vận hành hoặc Validation; minh họa các giá trị K trên Test thì **chỉ để mô tả**, không dùng để tối ưu mô hình.

In [ ]:
from src.experiment import precision_recall_at_k
actual = pred["y_true"].to_numpy().astype(int)
k_values = sorted(set(min(len(pred), k) for k in [50,100,200,500] if k > 0))
topk_rows = []
for model_name in ("dummy", "rf", "mlp"):
    probs = pred["p_"+model_name].to_numpy()
    for k in k_values:
        topk_rows.append({"model": model_name, **precision_recall_at_k(actual, probs, k)})
topk = pd.DataFrame(topk_rows)
display(topk)
OUT_TABLES.mkdir(parents=True, exist_ok=True)
topk.to_csv(OUT_TABLES / "topk_analysis.csv", index=False)
print("Top-K độc lập cho cả ba mô hình; không dùng Test để chọn K hoặc ngưỡng.")

## 5.4 Thảo luận – so sánh mô hình & hạn chế

Cần trả lời:
- RF (Ngọc Lan) vs MLP (Tấn Lên) khác nhau thế nào về AP, Precision, Recall, F1 và FP? Chênh lệch có đáng kể trong thực tiễn không?
- Số cảnh báo sai của từng mô hình thay đổi như thế nào khi ưu tiên Top-K?
- Model mắc lỗi nhiều ở nhóm nào? Có thể do ít dữ liệu hoặc đặc trưng không đầy đủ?
- Những hạn chế: **delivered-only selection bias**, dữ liệu cũ (2016–2018), địa lý Brazil, không tự áp dụng kết quả cho sàn Việt Nam 2026, feature availability chưa xác minh hết.

**Không diễn giải AP là tỷ lệ phần trăm đơn đúng.** AP đo chất lượng xếp hạng Precision–Recall.

## 5.5 Checklist bàn giao
- [ ] Metrics Test cho Dummy/RF/MLP, có hỗ trợ số mẫu và nhãn.
- [ ] Confusion matrix và PR Curve từ dữ liệu thật.
- [ ] FP, FN, Precision@K, Recall@K và phân tích trade-off.
- [ ] Có bảng lỗi theo nhóm đủ kích thước mẫu, nhận xét trung lập.
- [ ] Có liên kết repo/commit của file dự đoán và ngưỡng Validation.
- [ ] Hoàn thành `reports/chapters/CH05_RESULTS.md`, gửi bảng/hình cho Tấn Lên.

**Cách trình bày:** cho thấy bảng kết quả trước, giải thích chỉ số theo góc nhìn vận hành, rồi phân tích sai và hạn chế; tránh chỉ khen mô hình có AP cao nhất.
- [ ] Đã tái dự đoán Test từ RF Ngọc Lan + Dummy Trúc Chi và MLP Tấn Lên và đối chiếu xác suất với file bàn giao.
- [ ] Top-K của cả 3 mô hình đã tính độc lập, không lựa chọn threshold bằng Test.